# BPS Input Products Automatic Download

"""
SPDX-License-Identifier: MIT

Copyright (c) 2026 [Serco Italia SpA]
"""


This Notebook shows how to download all the necessary input products to run the BPS from Coding, given a single Biomass Level 0S (RAW_0S) product ID and the specified path in which you want to download the files.

To search for a Biomass L0S product ID you can refer to the [ESA MAAP Explorer](https://explorer.maap.eo.esa.int/). 

Once the desired product is found, you can just copy the ID, e.g. BIO_S3_RAW__0S_20260409T035949_20260409T040146_T_G01_M02_C06_T018_F001_01_DPLGMJ.



The notebook is divided into two sections:

1. Products download
2. NetCDF file generation from the products downloaded in section 1



## 📋 What gets downloaded

Given a RAW_0S product ID, the following matching products are automatically retrieved
for each acquisition:

| Product | Description |
|---|---|
| `BIO_*_RAW__0S_*` | Raw Single Look Complex — the reference product |
| `BIO_*_RAW__0M_*` | Raw Monitoring — matching acquisition |
| `BIO_AUX_ORB_*` | Orbit auxiliary data |
| `BIO_AUX_ATT_*` | Attitude auxiliary data |
| `BIO_AUX_TEC_*` | TEC auxiliary data (full day) |

---

## 📁 Output folder structure

All products are extracted into the `Inputs/` subfolder of the specified output path,
which is the structure expected by `BPS_Run*.ipynb`:

### Import the necessary functions

Ensure that *BPS_inputs_download.py* is in the same folder of this notebook.

In [ ]:
from BPS_inputs_download import get_raw_repeat_cycle, get_raws_major_cycle

### Section 1 — Single Repeat Cycle Download

In the following example, given the RAW_0S ID and the desired output path, all the corresponding RAW_0M, AUX_ORB, AUX_ATT and AUX_TEC products (and the RAW_0S itself) will be downloaded.

The folder specified in *output_path* will be automatically generated if it does not exist. The matching products will be downloaded in *.zip* format and extracted into the subfolder *Inputs*. Finally, the *.zip* products will be deleted to save memory.

Example:
```
/path/to/download/
|
|--- BIO_S3_RAW__0S_20260409T035949_20260409T040146_T_G01_M02_C06_T018_F001_01_DPLGMJ.zip (deleted)
|--- BIO_S3_RAW__0M_20260409T035214_20260409T040821_T_G01_M02_C06_T018_F____01_DPLOTG.zip (deleted)
|--- BIO_AUX_ATT____20260409T035159_20260409T040820_01_DPLOVM.zip (deleted)
|--- BIO_AUX_ORB____20260409T035159_20260409T040820_01_DPLOVM.zip (deleted)
|--- BIO_AUX_TEC____20260409T000000_20260409T235959_01_DPL1LQ.zip (deleted)
|--- Inputs/
    |--- BIO_S3_RAW__0S_20260409T035949_20260409T040146_T_G01_M02_C06_T018_F001_01_DPLGMJ
    |--- BIO_S3_RAW__0M_20260409T035214_20260409T040821_T_G01_M02_C06_T018_F____01_DPLOTG
    |--- BIO_AUX_ATT____20260409T035159_20260409T040820_01_DPLOVM
    |--- BIO_AUX_ORB____20260409T035159_20260409T040820_01_DPLOVM
    |--- BIO_AUX_TEC____20260409T000000_20260409T235959_01_DPL1LQ
```
So at the end you will have:
```
/path/to/download/
|--- Inputs/
    |--- BIO_S3_RAW__0S_20260409T035949_20260409T040146_T_G01_M02_C06_T018_F001_01_DPLGMJ
    |--- BIO_S3_RAW__0M_20260409T035214_20260409T040821_T_G01_M02_C06_T018_F____01_DPLOTG
    |--- BIO_AUX_ATT____20260409T035159_20260409T040820_01_DPLOVM
    |--- BIO_AUX_ORB____20260409T035159_20260409T040820_01_DPLOVM
    |--- BIO_AUX_TEC____20260409T000000_20260409T235959_01_DPL1LQ
```
Use this when you want to process a single pass over the area of interest — for example to run the L1F and L1 chain only.

In [ ]:
# USAGE FOR SINGLE CYCLE DOWNLOAD

ID = 'BIO_S3_RAW__0S_20260409T035949_20260409T040146_T_G01_M02_C06_T018_F001_01_DPLGMJ'
output_path = '/home/jovyan/BPS_test' # /home/jovyan/ is the default home directory for Jupyter

L0_S, L0_M, BIO_AUX_ATT, BIO_AUX_ORB, BIO_AUX_TEC = get_raw_repeat_cycle(ID, output_path)

### Section 2 — NetCDF file generation

Starting from a couple of input BIOMASS L0S / L0M products, plus corresponding auxiliary files, it will be generated an output extracted L0 product in NetCDF format.


the extracted NetCDF product contains:

- SAR RAW data, divided per polarization, with the corresponding timing metadata
- internal calibration data, divided per polarization, with the corresponding timing metadata
- on-board temperature monitoring data
- platform data (satellite position, velocity and attitude quaternions)
- instrument data (carrier frequency and transmitted pulse)
- main product metadata

#### Pre-requisites:
1. Download the AUX_INS file (and unzip it) into the same folder of all the other products, i.e. .../Inputs/. You can download the AUX_INS file from the [Biomass Disc page](https://biomass-disc.info/release_note).
2. bps-l0_extraction_tool shall be installed within your Conda environment. You can comment out the line: *!conda install -y biomass-bps::bps-l0_extraction_tool* if you already installed it.
3. The destination folder in which to save the NetCDF file must already exist before launching the tool

> **Note:**
> 
> If you installed bps-l0_extraction_tool in the previous session, after the server restart you will need to install it again, since sessions are not persistent. If you want to persistently install a package within your Conda environment, you need to create a new environment and a new kernel. See the [related documentation](https://portal.maap.eo.esa.int/ini/kb/books/collaborative-environment-pal/page/create-and-managing-environments).
> 
> 
> If you cannot run the *get_NetCDF* function, check which kernel you are running with your notebook.
>
> 
> 


The function *get_NetCDF* takes as input:
1. l0_downloaded_path: the path of the previously downloaded files (*output_path*)
2. netcdf_path: the path where you want to download your NetCDF file
3. start: start time of interest
4. stop: stop time of interest


All the inputs are mandatory except for start and stop times of interest.
If they are provided, only the portion of the L0S product inside this time window is extracted.
If they are not provided, the entire L0S product is extracted.



> If you do not want to provide any start or stop time of interest, just put 'None' within the function input parameters, e.g. get_NetCDF(l0_downloaded_path, netcdf_path, start=None, stop=None)
> 
>The NetCDF output folder should already exists when launching the function

The output product is a single NetCDF file with the same name of input L0S product and with .nc extension.

In [ ]:
!conda install -y biomass-bps::bps-l0_extraction_tool > /dev/null 2>&1
  
from BPS_inputs_download import get_NetCDF

l0_downloaded_path = output_path # Path where the required inputs are
netcdf_path = '/home/jovyan/BPS_test/Outputs' # Path where the .nc output file will be saved (it should already exist)

start = '21-NOV-2025 09:52:55.000000'
stop = '21-NOV-2025 09:53:10.000000'

#get_NetCDF(l0_downloaded_path, netcdf_path, start, stop)
get_NetCDF(l0_downloaded_path, netcdf_path, start=None, stop=None)